# 🚀 ENSEMBLE LEARNING: SPACESHIP TITANIC

## 1. Giới thiệu
Bộ dữ liệu **Spaceship Titanic** yêu cầu chúng ta dự đoán xem một hành khách có bị dịch chuyển sang không gian đa chiều (Transported) hay không, dựa trên các thông tin cá nhân, vị trí phòng ngủ (Cabin), và số tiền họ chi tiêu trên tàu.

Trong Notebook này, chúng ta sẽ thực hiện:
1. **Khám phá và Tiền xử lý dữ liệu (EDA & Preprocessing):** Xử lý missing values, tạo các đặc trưng mới (Feature Engineering) từ `Cabin` và `PassengerId`.
2. **Xây dựng mô hình:** Sử dụng các kỹ thuật Bagging (Random Forest), Boosting (XGBoost, LightGBM, CatBoost).
3. **Ensemble Learning:** Sử dụng kỹ thuật `Voting Classifier` (hoặc `Stacking`) để kết hợp sức mạnh của các mô hình, nhằm đẩy độ chính xác (Accuracy) trên tập Test lên mức > 80%.

In [12]:
# Cài đặt các thư viện cần thiết (nếu chạy trên Colab)
!pip install catboost lightgbm xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')

In [13]:
# Load dữ liệu (thay đổi đường dẫn nếu cần)
train_df = pd.read_csv('C:\\Users\\Vinh\\Desktop\\mliot-pyml-2026-hw\\week06\\dataset-spaceship-titanic\\train.csv')
test_df = pd.read_csv('C:\\Users\\Vinh\\Desktop\\mliot-pyml-2026-hw\\week06\\dataset-spaceship-titanic\\test.csv')

print(f"Kích thước tập Train: {train_df.shape}")
print(f"Kích thước tập Test: {test_df.shape}")

# Xem trước vài dòng dữ liệu
train_df.head()

Kích thước tập Train: (8693, 14)
Kích thước tập Test: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## 3. Tiền xử lý dữ liệu & Feature Engineering
Kỹ thuật trích xuất đặc trưng.
1. **PassengerId:** Có định dạng `gggg_pp`. Trong đó `gggg` là nhóm đi cùng nhau. Ta sẽ tạo ra đặc trưng `Group_Size` (kích thước nhóm).
2. **Cabin:** Có định dạng `deck/num/side`. Ta sẽ tách cột này thành 3 cột riêng biệt là `Deck`, `Num`, và `Side`.
3. **Chi tiêu (Expenditure):** Tính tổng chi tiêu trên tàu (RoomService, FoodCourt, ShoppingMall, Spa, VRDeck) thành một cột `Total_Spent`. Nếu hành khách đang ngủ đông (`CryoSleep == True`), chi tiêu của họ chắc chắn là 0.
4. **Xử lý Missing Values:** Điền giá trị trung vị (median) cho dữ liệu số, và giá trị xuất hiện nhiều nhất (mode) cho dữ liệu phân loại.

In [14]:
def preprocess_data(df):
    df = df.copy()
    
    # 1. Trích xuất thông tin từ PassengerId
    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])
    group_counts = df['Group'].value_counts().to_dict()
    df['Group_Size'] = df['Group'].map(group_counts)
    
    # 2. Xử lý cột Cabin (Tách thành Deck, Num, Side)
    df['Cabin'].fillna('Z/9999/P', inplace=True) # Z/9999/P coi như ẩn số
    df['Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
    df['Num'] = df['Cabin'].apply(lambda x: int(x.split('/')[1]))
    df['Side'] = df['Cabin'].apply(lambda x: x.split('/')[2])
    
    # 3. Xử lý các cột chi tiêu và CryoSleep
    exp_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    
    # Nếu CryoSleep là True, không thể tiêu tiền -> Điền 0
    for col in exp_cols:
        df.loc[df['CryoSleep'] == True, col] = 0
        
    # Điền NA cho chi tiêu bằng trung vị
    for col in exp_cols:
        df[col].fillna(df[col].median(), inplace=True)
        
    # Tạo đặc trưng tổng chi tiêu
    df['Total_Spent'] = df[exp_cols].sum(axis=1)
    df['Is_Spender'] = (df['Total_Spent'] > 0).astype(int)
    
    # Điền NA cho các cột Categorical
    cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
    for col in cat_cols:
        df[col].fillna(df[col].mode()[0], inplace=True)
        
    df['Age'].fillna(df['Age'].median(), inplace=True)
    
    # Xóa các cột không cần thiết cho mô hình
    cols_to_drop = ['PassengerId', 'Cabin', 'Name', 'Group']
    df = df.drop(columns=cols_to_drop)
    
    return df

# Áp dụng hàm tiền xử lý cho cả 2 tập
train_clean = preprocess_data(train_df)
test_clean = preprocess_data(test_df)

# Mã hóa các biến Categorical bằng LabelEncoder
categorical_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
le = LabelEncoder()

for col in categorical_features:
    train_clean[col] = le.fit_transform(train_clean[col].astype(str))
    test_clean[col] = le.transform(test_clean[col].astype(str))

# Tách biến mục tiêu (Target)
X = train_clean.drop(columns=['Transported'])
y = train_clean['Transported'].astype(int)
X_test = test_clean

print("Pre-processing hoàn tất. Kích thước tập huấn luyện:", X.shape)

Pre-processing hoàn tất. Kích thước tập huấn luyện: (8693, 16)


## 4. Huấn luyện mô hình (Benchmark)
Định nghĩa 4 mô hình độc lập đại diện cho các kỹ thuật Ensemble:
*   **Bagging:** Random Forest
*   **Boosting:** XGBoost, LightGBM, CatBoost

Sử dụng `K-Fold Cross Validation` để đánh giá và so sánh (benchmark) hiệu suất của các mô hình này trên tập Train trước khi kết hợp chúng lại.

In [15]:
# Khởi tạo các mô hình cơ sở với tham số tùy chỉnh nhẹ để tránh Overfitting
rf_model = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42)
xgb_model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42)
lgbm_model = LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42)
cat_model = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05, verbose=0, random_state=42)

models = {
    'Random Forest (Bagging)': rf_model,
    'XGBoost (Boosting)': xgb_model,
    'LightGBM (Boosting)': lgbm_model,
    'CatBoost (Boosting)': cat_model
}

# Đánh giá bằng Cross Validation
print("=== KẾT QUẢ BENCHMARK CÁC MÔ HÌNH (5-Fold CV) ===")
for name, model in models.items():
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f"{name}: Mean Accuracy = {cv_scores.mean():.4f} (Std = {cv_scores.std():.4f})")

=== KẾT QUẢ BENCHMARK CÁC MÔ HÌNH (5-Fold CV) ===
Random Forest (Bagging): Mean Accuracy = 0.7902 (Std = 0.0255)
XGBoost (Boosting): Mean Accuracy = 0.7835 (Std = 0.0316)
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000672 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1895
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv

## 5. Áp dụng kỹ thuật Stacking / Voting
Dựa vào kết quả Benchmark ở trên, ta thấy các mô hình Boosting (đặc biệt là CatBoost và LightGBM) thường cho kết quả cao nhất (khoảng ~80-81%).

Để mô hình tổng quát hóa tốt hơn nữa trên tập Test, ta dùng kỹ thuật **Soft Voting Classifier**. Thay vì để 1 mô hình quyết định, Voting Classifier sẽ lấy trung bình cộng xác suất dự đoán (probabilities) của tất cả các mô hình mạnh nhất rồi mới đưa ra quyết định cuối cùng.

In [16]:
# Tạo ra một Voting Ensemble kết hợp 3 mô hình Boosting mạnh nhất
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('cat', cat_model)
    ],
    voting='soft' # 'soft' sử dụng xác suất dự đoán để vote, thường tốt hơn 'hard'
)

# Huấn luyện mô hình Ensemble trên toàn bộ tập train
print("Đang huấn luyện Voting Ensemble Model...")
voting_clf.fit(X, y)
print("Huấn luyện hoàn tất!")

Đang huấn luyện Voting Ensemble Model...
[LightGBM] [Info] Number of positive: 4378, number of negative: 4315
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000759 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1896
[LightGBM] [Info] Number of data points in the train set: 8693, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503624 -> initscore=0.014495
[LightGBM] [Info] Start training from score 0.014495
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

## 6. Dự đoán và xuất file Submit
Sử dụng mô hình Voting Ensemble vừa huấn luyện để dự đoán trên tập `test.csv`. Kaggle yêu cầu cột `Transported` mang giá trị Boolean (True/False).

In [17]:
# Dự đoán trên tập test
preds = voting_clf.predict(X_test)

# Chuyển đổi nhãn (0, 1) về lại (False, True)
preds_bool = [True if x == 1 else False for x in preds]

# Tạo dataframe để submit
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': preds_bool
})

# Lưu ra file CSV
submission.to_csv('submission.csv', index=False)
print("Đã lưu file submission.csv")
submission.head()

Đã lưu file submission.csv


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
